In [ ]:
import sys
import warnings
from functools import partial

import pandas as pd
import torch
import yaml
from tabpfn import TabPFNRegressor

sys.path.append("..")
from src.data_generation.data_preperation import data_preparation, make_stratified_eval_set
from src.model.finetune import finetune
from src.evaluation.surface_eval import eval_surfaces
cfg = yaml.safe_load(open("../config.yaml"))

warnings.filterwarnings("ignore", message="Running on CPU with more than")

In [ ]:
N_CONTEXT = 20
RUN_NAME = "ssvi_uniform_context_3_30"
data_provider = partial(data_preparation, cfg, n_context=N_CONTEXT)

In [ ]:
data_provider = partial(data_preparation, cfg, n_context=(3, 30)) 
val_data = make_stratified_eval_set(cfg, n_surfaces=20, context_sizes=[5, 10, 20, 40])

finetune(data_provider, run_name=RUN_NAME, n_epochs=1, n_surfaces_per_epoch=1,
         batch_size=4, val_data=val_data, val_every=5)

In [ ]:
N_ESTIMATORS = 1 # finetuned with 1
baseline = TabPFNRegressor(
    n_estimators=N_ESTIMATORS, inference_config={"FINGERPRINT_FEATURE": False},
)

finetuned = TabPFNRegressor(
    fit_mode="fit_preprocessors", n_estimators=N_ESTIMATORS,
    inference_config={"FINGERPRINT_FEATURE": False},
)

finetuned._initialize_model_variables()
finetuned_state = torch.load(f"../checkpoints/{RUN_NAME}/best.pt", map_location="cpu")
finetuned.model_.load_state_dict(finetuned_state)

In [ ]:
N_CONTEXT_VALUES = [5, 8, 10, 15, 20, 40]
N_TEST_SURFACES_SWEEP = 50

sweep_results = []
for n_ctx in N_CONTEXT_VALUES:
    tr, te = data_preparation(cfg, N_TEST_SURFACES_SWEEP, n_ctx)

    b = eval_surfaces(baseline, tr, te, cfg)
    f = eval_surfaces(finetuned, tr, te, cfg, reload_state=finetuned_state)

    sweep_results.append((n_ctx, b, f))
    print(f"N_CONTEXT= {n_ctx} done")

rows, rows_arb = [], []
for n_ctx, b, f in sweep_results:
    rows.append({
        "N_CONTEXT": n_ctx,
        "Base MAE": b[0], "FT MAE": f[0], "MAE Δ%": (f[0] - b[0]) / b[0] * 100,
        "Base MAPE%": b[1], "FT MAPE%": f[1], "MAPE Δ%": (f[1] - b[1]) / b[1] * 100,
    })
    rows_arb.append({
        "N_CONTEXT": n_ctx,
        "Base cal%": b[2] * 100, "FT cal%": f[2] * 100,
        "Base bfly%": b[3] * 100, "FT bfly%": f[3] * 100,
    })

In [ ]:
df = pd.DataFrame(rows).set_index("N_CONTEXT")
df.style.format({
    "Base MAE": "{:.4f}", "FT MAE": "{:.4f}", "MAE Δ%": "{:.1f}",
    "Base MAPE%": "{:.2f}", "FT MAPE%": "{:.2f}", "MAPE Δ%": "{:.1f}",
})

In [ ]:
df_arb = pd.DataFrame(rows_arb).set_index("N_CONTEXT")
df_arb.style.format("{:.0f}")